# OpenAI SDK Agentic AI: AsyncOpenAI & Agent Basics

## 📘 Overview

This notebook is a **progressive, hands-on tutorial** that builds up from raw API calls to a full AI Agent. You'll learn:

```mermaid
flowchart LR
    A["🔑 Load .env<br/>API Key"] --> B["📡 Chat Completion<br/>Direct API Call"]
    B --> C["⚡ AsyncOpenAI<br/>Non-blocking I/O"]
    C --> D["🌉 OpenAIChatCompletionsModel<br/>Bridge to Agent SDK"]
    D --> E["🤖 Agent + Runner<br/>AI Persona & Execution"]
    E --> F["🔄 Multi-turn<br/>Conversation"]
```

| Stage | What You'll Do | Key Concept |
|---|---|---|
| **Part 1** | Raw `chat.completions.create()` call | The foundation — every Agent call is ultimately a chat completion |
| **Part 2** | Switch to `AsyncOpenAI` | Non-blocking I/O for concurrent requests |
| **Part 3** | Bridge into the Agent SDK | `OpenAIChatCompletionsModel` wraps any OpenAI-compatible endpoint |
| **Part 4** | Define & run an Agent | `name` + `instructions` + `model` = AI persona |
| **Part 5** | Multi-turn conversation | Maintaining context across turns |

### Environment

- **Conda env**: `agentic_ai` (Python 3.13)
- **API Key**: `DEEPSEEK_API_KEY` stored in `.env` file (see prerequisites below)
- **Packages**: `openai>=1.40`, `openai-agents>=0.0.6`, `python-dotenv`
- **Tutorial path**: `tutorials/02-agentic-ai/01-AsyncOpenAI-Agent-Basics.ipynb`

## ⚙️ Prerequisites & Environment

### `.env` File (REQUIRED)

Your **`DEEPSEEK_API_KEY` is stored in a `.env` file** located at `tutorials/02-agentic-ai/.env`. This file is **never committed to git** — it keeps your API key secure.

```bash
# tutorials/02-agentic-ai/.env
DEEPSEEK_API_KEY=sk-your-actual-deepseek-key-here
```

```mermaid
flowchart TD
    A[".env file on disk"] -->|"load_dotenv()"| B["os.environ"]
    B -->|"os.getenv('DEEPSEEK_API_KEY')"| C["AsyncOpenAI client"]
    C -->|"Bearer token"| D["https://api.deepseek.com/v1"]
```

> 🔑 **Get your key**: Sign up at [platform.deepseek.com](https://platform.deepseek.com) → API Keys → Create. The `.env` file already exists in this tutorial folder — just fill in your key.

### Conda Environment

```bash
conda activate agentic_ai
```

### Required Packages

| Package | Version | Role |
|---|---|---|
| `openai` | >=1.40 | OpenAI/DeepSeek API client (sync + async) |
| `openai-agents` | >=0.0.6 | Agent, Runner, handoff, tool abstractions |
| `python-dotenv` | >=1.0 | Load `.env` → `os.environ` securely |

## 📦 Install & Verify

```bash
conda activate agentic_ai
pip install -r tutorials/02-agentic-ai/requirements.txt
```

> ✅ The `agentic_ai` conda environment already has these pre-installed. The cell below verifies your setup.

## Part 1 — Chat Completion API: The Foundation

Before using the Agent SDK, let's understand the **raw chat completion API** — the fundamental building block that every Agent call ultimately relies on.

```mermaid
sequenceDiagram
    participant You as 🧑 You (Python)
    participant SDK as 🐍 OpenAI SDK
    participant API as ☁️ DeepSeek API
    You->>SDK: client.chat.completions.create(<br/>model="deepseek-chat",<br/>messages=[...])
    SDK->>API: POST /v1/chat/completions<br/>(JSON payload with messages)
    API-->>SDK: { choices: [{ message: { content: "..." } }] }
    SDK-->>You: response.choices[0].message.content
```

### What happens under the hood:

1. You send a **list of messages** (system + user roles)
2. The API processes them and returns a **completion** (the model's reply)
3. The `content` field contains the text response

### The cell below will:

- Load `DEEPSEEK_API_KEY` from your `.env` file
- Call the chat completion API directly (synchronous mode)
- Print the model's response

In [7]:
# ============================================================================
# Part 1: Load Environment & Run a Raw Chat Completion
# ============================================================================
import os
from dotenv import load_dotenv

# --- Load .env ---
# Searches for .env in the current working directory and parent dirs.
# `override=True` ensures .env values win over existing env vars.
load_dotenv(override=True)

# --- Verify the API key loaded ---
api_key = os.getenv("DEEPSEEK_API_KEY")
if api_key:
    # Show first 6 and last 4 chars only — NEVER print the full key!
    print(f"✅ DEEPSEEK_API_KEY loaded: {api_key[:6]}...{api_key[-4:]}")
else:
    print("❌ DEEPSEEK_API_KEY NOT found! Check your .env file.")
    print("   Expected location: tutorials/02-agentic-ai/.env")

# ============================================================================
# Synchronous Chat Completion (the simplest possible call)
# ============================================================================
from openai import OpenAI  # Synchronous client — blocks until response arrives

# Create a sync client pointed at DeepSeek
# NOTE: base_url changed to https://api.deepseek.com (no /v1 suffix, as of Aug 2026)
sync_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

# Make the API call with the latest model: deepseek-v4-flash (DeepSeek-V4-Flash-0731)
print("\n📡 Calling DeepSeek chat completion API (model: deepseek-v4-flash)...")
response = sync_client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
        {"role": "user", "content": "What is the capital of France, and what's one famous landmark there?"}
    ],
    temperature=0.7,
    max_tokens=200
)

# Extract and print the result
answer = response.choices[0].message.content
print(f"\n🤖 DeepSeek Response:\n{answer}")
print(f"\n📊 Model used: {response.model}")
print(f"📊 Tokens used: {response.usage.total_tokens} "
      f"(prompt: {response.usage.prompt_tokens}, "
      f"completion: {response.usage.completion_tokens})")

✅ DEEPSEEK_API_KEY loaded: sk-007...32aa

📡 Calling DeepSeek chat completion API (model: deepseek-v4-flash)...

🤖 DeepSeek Response:
The capital of France is Paris. One famous landmark there is the Eiffel Tower.

📊 Model used: deepseek-v4-flash
📊 Tokens used: 148 (prompt: 109, completion: 39)


### ✅ What Just Happened?

```mermaid
flowchart TD
    A["load_dotenv()"] --> B["os.getenv('DEEPSEEK_API_KEY')"]
    B --> C["OpenAI(base_url='https://api.deepseek.com/v1')"]
    C --> D["client.chat.completions.create()"]
    D --> E["{ model, messages, temperature, max_tokens }"]
    E --> F["HTTP POST → DeepSeek API"]
    F --> G["response.choices[0].message.content"]
```

| Step | Detail |
|---|---|
| **Load key** | `python-dotenv` reads `.env` → `os.environ` |
| **Create client** | `OpenAI(api_key=..., base_url=...)` — sync, blocking client |
| **Call API** | `chat.completions.create(model, messages, ...)` |
| **Parse response** | `.choices[0].message.content` extracts the text |
| **Token tracking** | `.usage.total_tokens` shows cost |

> 💡 **This is the core primitive.** Every Agent call, every tool use, every handoff — it all boils down to `chat.completions.create()`. The Agent SDK is just a convenience layer on top.

In [10]:
# ============================================================================
# Part 2: AsyncOpenAI — Non-blocking Chat Completion
# ============================================================================
import asyncio
from openai import AsyncOpenAI  # Async client — non-blocking, await-based

# Create the async client (same api_key & base_url as before)
# NOTE: base_url = https://api.deepseek.com (no /v1, as of Aug 2026)
async_client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

# ============================================================================
# Async chat completion — identical API, just add `await`
# ============================================================================
async def async_chat_demo():
    print("📡 Sending async chat completion request...")
    response = await async_client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
            {"role": "user", "content": "Explain what 'async/await' means in Python in one sentence."}
        ],
        temperature=0.7,
        max_tokens=150
    )
    return response

# Run the async function
response = await async_chat_demo()

answer = response.choices[0].message.content
print(f"\n🤖 DeepSeek Response:\n{answer}")
print(f"\n📊 Tokens used: {response.usage.total_tokens}")

📡 Sending async chat completion request...

🤖 DeepSeek Response:
Async/await in Python lets you write concurrent, non-blocking code by defining coroutines with `async def` and pausing them at `await` points, allowing other tasks to run while waiting for I/O or other operations to complete.

📊 Tokens used: 177


### Sync vs Async: Why `AsyncOpenAI` Matters

```mermaid
sequenceDiagram
    participant Sync as 🔒 Sync (OpenAI)
    participant Async as ⚡ Async (AsyncOpenAI)
    participant API as ☁️ DeepSeek API

    Note over Sync,API: SYNCHRONOUS — one at a time
    Sync->>API: Request 1
    API-->>Sync: Response 1
    Sync->>API: Request 2
    API-->>Sync: Response 2
    Note over Sync: Total time = R1 + R2

    Note over Async,API: ASYNCHRONOUS — overlap I/O
    Async->>API: Request 1
    Async->>API: Request 2
    API-->>Async: Response 1
    API-->>Async: Response 2
    Note over Async: Total time ≈ max(R1, R2)
```

| | Sync (`OpenAI`) | Async (`AsyncOpenAI`) |
|---|---|---|
| **Execution** | Blocking — each call waits | Non-blocking — `await` yields to event loop |
| **Concurrent calls** | Need `threading` / `multiprocessing` | Native `asyncio.gather()` |
| **I/O efficiency** | Threads idle during network wait | Event loop switches tasks |
| **Best for** | Single requests, simple scripts | Agent workflows, parallel tools |

> 🔑 **Key insight**: LLM API calls are I/O-bound (network latency >> CPU time). Async lets you overlap multiple requests. This is critical when agents call tools in parallel.

## Part 3 — Agent SDK: Bridge & Agent Definition

### Architecture: From Async Client to AI Agent

```mermaid
flowchart TD
    subgraph "Your Code"
        A["AsyncOpenAI<br/>client"] --> B["OpenAIChatCompletionsModel<br/>bridge wrapper"]
        B --> C["Agent<br/>name + instructions + model"]
        C --> D["Runner.run()<br/>async execution"]
    end
    subgraph "DeepSeek Cloud"
        E["https://api.deepseek.com/v1<br/>chat/completions endpoint"]
    end
    A -.->|"HTTP POST"| E
    D --> F["result.final_output"]
```

The `openai-agents` SDK provides three key classes:

| Class | Role |
|---|---|
| `OpenAIChatCompletionsModel` | Wraps any OpenAI-compatible client into a unified model interface the SDK understands |
| `Agent` | Defines an AI persona: `name`, `instructions` (system prompt), and `model` |
| `Runner` | Orchestrates agent execution with `await`-based concurrency |

The chat-completions standard is supported by virtually all major LLM providers: DeepSeek, OpenAI, Minimax, Moonshot (Kimi), Zhipu (GLM), ByteDance (Doubao), Baidu (ERNIE), and local endpoints (vLLM, Ollama).

### Model Wrapper & Agent Creation

The cell below:
1. Wraps the `AsyncOpenAI` client with `OpenAIChatCompletionsModel`
2. Disables OpenAI's built-in tracing (unnecessary for DeepSeek)
3. Creates an Agent with a specific persona

In [9]:
# ============================================================================
# Part 3: Agent SDK — Bridge + Agent Definition
# ============================================================================
from agents import (
    Agent, Runner,
    OpenAIChatCompletionsModel,
    set_tracing_disabled
)

# --- Disable tracing (not needed for third-party endpoints) ---
set_tracing_disabled(True)

# --- Step 1: Wrap the async client with the bridge class ---
# OpenAIChatCompletionsModel adapts any OpenAI-compatible client
# so the Agent SDK can use it as a drop-in model.
model = OpenAIChatCompletionsModel(
    model="deepseek-v4-flash",   # DeepSeek-V4-Flash-0731 (latest, 1M context)
    openai_client=async_client   # Our AsyncOpenAI client from Part 2
)

# --- Step 2: Create an Agent ---
# An Agent = name + instructions (system prompt) + model
agent = Agent(
    name="Albert Einstein",
    instructions=(
        "You are an insightful academician with a talent for making complex "
        "ideas accessible. Explain concepts clearly, use analogies where "
        "helpful, and always be patient with learners."
    ),
    model=model,
)

print("✅ Agent created successfully!")
print(f"   Name: {agent.name}")
print(f"   Model: deepseek-v4-flash (DeepSeek-V4-Flash-0731)")
print(f"   Instructions: {agent.instructions[:80]}...")

✅ Agent created successfully!
   Name: Albert Einstein
   Model: deepseek-v4-flash (DeepSeek-V4-Flash-0731)
   Instructions: You are an insightful academician with a talent for making complex ideas accessi...


## Part 4 — Run the Agent

### Execution Flow

```mermaid
sequenceDiagram
    participant You as 🧑 Your Code
    participant Runner as 🏃 Runner
    participant Agent as 🤖 Agent
    participant API as ☁️ DeepSeek API

    You->>Runner: await Runner.run(agent, "Explain relativity")
    Runner->>Agent: Process with instructions + user query
    Agent->>API: POST /v1/chat/completions<br/>(system + user messages)
    API-->>Agent: { choices: [{ message: { content: "..." } }] }
    Agent-->>Runner: result
    Runner-->>You: result.final_output
```

- `Runner.run()` is **async** — use `await` to yield control during I/O
- The agent's `instructions` become the **system message**
- The user's input becomes the **user message**
- `result.final_output` contains the agent's text response

In [11]:
# ============================================================================
# Part 4: Run the Agent — Single-turn Query
# ============================================================================

async def run_agent_demo():
    """Send a single question to the agent and print the response."""
    print("🚀 Running agent with query: 'Explain the theory of relativity in simple terms.'")
    print("⏳ Waiting for DeepSeek API response...\n")

    result = await Runner.run(
        agent,
        "Explain the theory of relativity in simple terms."
    )

    print("=" * 60)
    print("🤖 Agent Response:")
    print("=" * 60)
    print(result.final_output)
    print("=" * 60)

# Execute
await run_agent_demo()

🚀 Running agent with query: 'Explain the theory of relativity in simple terms.'
⏳ Waiting for DeepSeek API response...

🤖 Agent Response:
Imagine you’re playing catch on a moving train.

From your point of view, the ball just goes up and down. But from someone standing on the platform, the ball also moves forward with the train. Both views are true at the same time. There is no single “absolute” answer to how the ball is moving — it depends on your frame of reference.

That’s the heart of relativity: **the laws of physics are the same for everyone, no matter how fast they’re moving**, but measurements like time, distance, and even mass can look different depending on how fast you’re going.

### Special Relativity (Einstein, 1905)

This is about objects moving at constant speeds in a straight line. The key idea is:

- The speed of light in a vacuum is always the same for everyone, no matter how fast you’re moving.
- Because of that, **time itself must stretch** to keep the speed of ligh

## Part 5 — Multi-turn Conversation

### Beyond Single-turn: Building Context

The raw chat completion API supports **conversation history** by appending to the `messages` list. This is how chat UIs maintain context:

```mermaid
sequenceDiagram
    participant User as 🧑 User
    participant Code as 🐍 Your Code
    participant API as ☁️ DeepSeek

    Note over User,API: Turn 1
    User->>Code: "What is the capital of France?"
    Code->>API: messages=[{user: "What is the capital of France?"}]
    API-->>Code: "The capital of France is Paris."
    Code-->>User: "The capital of France is Paris."

    Note over User,API: Turn 2 — previous exchange is now in messages
    User->>Code: "What's a famous landmark there?"
    Code->>API: messages=[<br/>  {user: "What is the capital of France?"},<br/>  {assistant: "The capital of France is Paris."},<br/>  {user: "What's a famous landmark there?"}<br/>]
    API-->>Code: "The Eiffel Tower is a famous landmark in Paris."
```

The cell below demonstrates a 2-turn conversation using the raw chat completion API.

In [12]:
# ============================================================================
# Part 5: Multi-turn Conversation with Chat History
# ============================================================================

async def multi_turn_demo():
    """Demonstrate a 2-turn conversation with context carried via messages list."""

    # Start with a system message that sets the assistant's tone
    messages = [
        {"role": "system", "content": "You are a concise tour guide. Keep answers brief."},
    ]

    # --- Turn 1 ---
    print("=" * 60)
    print("🔄 TURN 1")
    print("=" * 60)
    messages.append({"role": "user", "content": "What is the capital of France?"})

    response = await async_client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=messages,
        temperature=0.7,
        max_tokens=100
    )
    turn1_answer = response.choices[0].message.content
    print(f"🧑 User:   What is the capital of France?")
    print(f"🤖 Agent:  {turn1_answer}")
    print(f"📊 Tokens: {response.usage.total_tokens}")

    # --- Store assistant response for context ---
    messages.append({"role": "assistant", "content": turn1_answer})

    # --- Turn 2 ---
    print(f"\n{'='*60}")
    print("🔄 TURN 2")
    print("=" * 60)
    messages.append({"role": "user", "content": "What's a famous landmark there?"})

    response = await async_client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=messages,
        temperature=0.7,
        max_tokens=100
    )
    turn2_answer = response.choices[0].message.content
    print(f"🧑 User:   What's a famous landmark there?")
    print(f"🤖 Agent:  {turn2_answer}")
    print(f"📊 Tokens: {response.usage.total_tokens}")

    # --- Show full message history ---
    print(f"\n{'='*60}")
    print("📋 Full Message History (5 messages):")
    print("=" * 60)
    for i, msg in enumerate(messages):
        print(f"  [{i}] {msg['role']:10} | {msg['content'][:70]}...")

    print(f"\n💡 Notice: Turn 2 used {response.usage.prompt_tokens} prompt tokens "
          f"(vs {response.usage.prompt_tokens - response.usage.completion_tokens} new) "
          f"because the history was included!")

# Run
await multi_turn_demo()

🔄 TURN 1
🧑 User:   What is the capital of France?
🤖 Agent:  Paris.
📊 Tokens: 136

🔄 TURN 2
🧑 User:   What's a famous landmark there?
🤖 Agent:  The Eiffel Tower.
📊 Tokens: 185

📋 Full Message History (5 messages):
  [0] system     | You are a concise tour guide. Keep answers brief....
  [1] user       | What is the capital of France?...
  [2] assistant  | Paris....
  [3] user       | What's a famous landmark there?...

💡 Notice: Turn 2 used 114 prompt tokens (vs 43 new) because the history was included!


## Part 6 — Comparing the Approaches

```mermaid
flowchart TD
    subgraph Raw["🟢 Raw API (Part 1-2)"]
        direction LR
        R1["client.chat.completions.create()"] --> R2["Full control over messages"]
        R2 --> R3["Manual history management"]
    end
    subgraph Agent["🔵 Agent SDK (Part 3-4)"]
        direction LR
        A1["Agent + Runner.run()"] --> A2["System prompt via instructions"]
        A2 --> A3["Auto history via Runner"]
    end
    Raw -->|"Both use"| API["Same DeepSeek API endpoint"]
    Agent -->|"Both use"| API
```

| Aspect | Raw Chat Completion API | Agent SDK |
|---|---|---|
| **Setup complexity** | Minimal — just a client | Medium — model bridge + agent definition |
| **Control** | Maximum — you manage everything | Less — SDK manages session/history |
| **System prompt** | Explicit `{"role": "system", ...}` | `instructions=` parameter |
| **History** | You append to `messages[]` | `Runner` handles it automatically |
| **Multi-agent** | You orchestrate manually | Built-in `handoff`, `as_tool` |
| **Best for** | Simple chatbots, fine-grained control | Complex agent workflows, tool use |

> 💡 **The Agent SDK IS the raw API underneath.** `Runner.run()` ultimately calls `chat.completions.create()` — the SDK just adds structure, session management, and multi-agent orchestration on top.

In [13]:
# ============================================================================
# Bonus: Agent with a Different Persona
# ============================================================================
# Quick test — create a second agent with a completely different role
# to show how instructions shape behavior.

coder_agent = Agent(
    name="Code Reviewer",
    instructions=(
        "You are a senior software engineer performing code review. "
        "Be rigorous, point out potential bugs, and suggest improvements. "
        "Keep answers under 200 words."
    ),
    model=model,  # Reuse the same model wrapper
)

async def bonus_test():
    print("🚀 Running code-review agent...\n")
    result = await Runner.run(
        coder_agent,
        "Review this Python snippet: `def divide(a, b): return a / b`"
    )
    print(result.final_output)

await bonus_test()

🚀 Running code-review agent...

The function works for basic numeric division but has two key issues:

1. **Division by zero** raises `ZeroDivisionError` with no clear feedback to callers.
2. **Type safety** – passing non-numeric types causes cryptic `TypeError`, and the function lacks type hints.

Improvements:

- Add type hints and handle zero gracefully (e.g., return `None` or raise a domain-specific exception).
- For Python 2 compatibility, add `from __future__ import division`, but preferably target Python 3 only.

Suggested rewrite:

```python
def divide(a: float, b: float) -> float | None:
    """Returns a / b, or None if b is zero."""
    if b == 0:
        return None
    return a / b
```

If raising is preferred, document it clearly:

```python
def divide(a: float, b: float) -> float:
    """Divides a by b. Raises ZeroDivisionError if b is zero."""
    return a / b
```

Add docstring and explicit error handling to make the function robust and self-documenting.


## 🎯 Summary: Your Learning Journey

```mermaid
flowchart LR
    A["🔑 .env Loaded"] -->|"Part 1"| B["📡 Raw Chat API<br/>sync call works"]
    B -->|"Part 2"| C["⚡ AsyncOpenAI<br/>non-blocking I/O"]
    C -->|"Part 3"| D["🤖 Agent Created<br/>with persona"]
    D -->|"Part 4"| E["🏃 Runner.run()<br/>async execution"]
    E -->|"Part 5"| F["🔄 Multi-turn<br/>conversation"]
    F -->|"Bonus"| G["🎭 Code Reviewer<br/>different persona"]
```

### You now understand:

| Concept | How It Works |
|---|---|
| **`.env` loading** | `python-dotenv` → `os.getenv("DEEPSEEK_API_KEY")` — secure, never hardcoded |
| **Chat completion** | `client.chat.completions.create(model, messages)` — the universal LLM API |
| **Async I/O** | `AsyncOpenAI` + `await` — overlap multiple API calls without threads |
| **Model bridge** | `OpenAIChatCompletionsModel` — adapts any provider for the Agent SDK |
| **Agent persona** | `Agent(name, instructions, model)` — role-based AI with system prompt |
| **Agent execution** | `await Runner.run(agent, query)` — async orchestration |
| **Conversation** | Append to `messages[]` — context carries across turns |

### Next Tutorials

- **[02-WebSearch-Agent.ipynb](02-WebSearch-Agent.ipynb)** — Give agents web search tools
- **[03-Handoff-and-as_tool.ipynb](03-Handoff-and-as_tool.ipynb)** — Multi-agent handoffs & agents-as-tools
- **[04-Replacement-01-MultiAgent.ipynb](04-Replacement-01-MultiAgent.ipynb)** — Advanced multi-agent orchestration

---

For inquiries: yucongcai_business@outlook.com (business) | yucongcai_research@outlook.com (research)

---

## 📝 Version Log

| Version | Date | Change |
|---|---|---|
| v1.0 | 2026-08-03 | Initial rebuild from `assets/previous-resources/` |
| v1.1 | 2026-08-04 | Added detailed markdown, Python comments, fixed model overwrite |
| v2.0 | 2026-08-04 | **Major restructure**: added raw Chat Completion API demo (Part 1), Async comparison (Part 2), multi-turn conversation (Part 5), mermaid diagrams throughout, `.env` security emphasis, comparison table (Part 6), code-review bonus agent |

### v2.0 changes (2026-08-04)

| Change |
|---|
| **Added Part 1**: Raw `chat.completions.create()` demo — the foundational API before the Agent SDK |
| **Added Part 2**: `AsyncOpenAI` async chat completion with sync vs async comparison (mermaid sequence diagram) |
| **Added Part 5**: Multi-turn conversation demo with message history |
| **Added Part 6**: Side-by-side comparison of raw API vs Agent SDK approaches |
| **Added**: 6 mermaid diagrams (learning path, .env flow, request sequence, sync vs async, agent execution, conversation history) |
| **Added**: Bonus agent test (Code Reviewer persona) |
| **Enhanced**: `.env` explanation with security best practices (never print full key, masked verification) |
| **Enhanced**: All code cells now print verification output (token counts, masked keys, status messages) |
| **Removed**: Placeholder alternative provider cells (irrelevant for tutorial flow) |